# Kaggriculture v18

Adaptive counter-policy successor to v17. Targets the v17 five-hire/sheep-heavy family while retaining the v16 replay-hardening ideas. The submitted main.py uses only live observation/private inventory: no replay lookup or seed-specific branches.


In [ ]:
%%capture
!pip install --upgrade "kaggle-environments==1.32.7"


## Build v18 entrypoint
The source is embedded compressed so the notebook is self-contained. Key deltas from v17: four-hire asymmetric opening, continuing public-state opponent classification, opponent-saturation penalties for livestock/crops, price-aware inventory holding, and earlier distance-aware final liquidation.


In [ ]:
import base64, gzip, pathlib
blob = '''H4sIAL5bomoC/608a3PixrLf+RU6SZ0CwQhLYGwvWK5yvCTryu7aZTvZyqFUlAzCKMYSBwk/smf/++3ueWhGEl4ndf3BwGimp6enp9+jH3744dfw7m4Tz7arfLuJrEfvaGiF83Cdx4+RNUu3SR5tnHW6imcv1iLdQIdDZxE+xKsXK0zm1iZar8IXZxlu5lESzS381W00bpaRJQbdR9E6w2HNDOA9rMNZbm3SbR4ndwThIdzcRznCjmZhhs3Mut1Cn+ghfYwyK19GjTSJnGyZ5la6XsP3JLdmqzDL4kU8C/M4TaynKLxPoizrWtZ5bm0zGBcmVpi9PDxEOawOwG83zjLeAKx1lODcrSxFrKx5Cp0TgB3DCvJ48QLDYaQV54j1gVxsHm7uotxmgNcsvUvivwRqC6ATAbayJazUWUbh4wsBFgMXm/TBWm9vgRhWlod5xKzZMk0RxSR6slYwPsvT2b11+9IAWtzFSbiSRHkMV9uIqKQWnoWwT7RoZoXr9SoGOLNNusZ/T3NcGKwvXOXQzho4EubcwFI2cXbvhE8hILqK/7uN55xwUbgBEBsLvsJqrAXNPld7mG1vH+I8h40N73B2omyaID2WEeFupbdZtHnk0HA+pNt6E0NLZMXJI4xKNwAOoMHUMZJaMI21StP77ZpZWRTNnWwdzXA/rdtoGT7G6YYBMqsoe8ny6MEKZzPYXQYUy5/SzX1D/gaGzIH4c2cNi3yx5hEsHrZxBhP+8MMPjfhhnW5yoGa+bDTOri4ur/2vDav55cP49KY5bHku6w1Yq7XP9m1m4z94eHZ6dXWBT3su6+PTPuvj0z49vbn4dHpzAU8HLjtw4ekR69ms9Y7+Izz88ODDZp5HI65vrk6//DS+uvqDZnSZ18Nxsm+Pf+zzjwM+8oBGfhp/vPgMg44QTTHmAFHxXPtb4/Tz+afTj7Cg5tnFF+i1D6Cbn84//tpkgBOi27z+MB5fEq7w6MvFxccmO2B9XGfzl4uL6zE86uOj8S+/NNk+Q7S/Na4/EJmaP53+OiacBbl4Nxh6ef6f/5xOsRs+5DNKujDRmfD/6eq3z2cfpteXRE4+iwKmkQVg/nF69Xl6fXNxNaYJCVX+QZDOz8bTs6vx6Sc1rTZcrlpN3bwc30zPTn8mWGI7mfzC9+TTxcXNh/PxK+Co38+nV5/GV9fTT6dXv45vdFoosGrh+oK+Na4ufrsZX/stwVzwv0//cXuRpbCNGKvVg+84mfhrtQb0cEAPB2LAAbXB5tGvQ/p1SL+0gX3iIdr6lkdQcXbgQmBS+GULlpleXpx/voEtXpNAR1mMJ9XiKFPbGn/TA+SH8XtfLmMgFjOg7wOAeXl6fe1PmvjRDBoXl5ewjX98HPufQVw1Go15tEBpMItaMZxjVkgEe0iYz33V4sDhcF1qjRcWdvd9eVQ3EUi9BA7yM6ysN3AGbTzT3VV6561bc3tP/oJT4Nk2jp+f+K4VrbLI6g069Dj77yZvOXNbnwHX2arZTGL1oaLtbZhFLF8CRe6WDETWPH1K8OBJUQFyYh9OqMe6hzYrZAQIiB6cr+4+68K5JaAkOPr9HjXCcZsgFoGaaOuHtxkuSEymHuCKjl1JB8Sn3fI623W7te0ctZEuLts6nt1uA8OoUQbZxCAHkW8XFNnanGCIyL98QpLTrRgFO7OL3v1+37ZLe6ZLPAMDkHyO133Xa89LWwQPOkfd/fbOfQKoQhqWOMF1uq7XnlcgwpPOwMDaMdAGNinDx1NfwvfAdXqDg/Z8z+v1yigfuJ13BwbG1KvMvyjDSki7rnOw77ZbMMAd0I6VcHfdzv5u3D33gM9i4gpQu0hZcejmcZa3QnZry8mRscKJGzi38M/u8J8e/vQCm4agvcWH8KP57GNXBweN5i8+dnRwiFwiseqzfcJ59sUm9T9/lhNOmuPT65smre35RKyt+WUMbQrE/KXofQ3y5wPv/qK6f764gsZAXy+KGsI3AeMFbKcpGF7z1lot9CFOWii02H304q/Ch9t5aGVDoseaZTZfaw62RWvBilGLSRPbsmYwWcMa8b8bBLxvmoerrCWMGkEdmBlMZn8ez3L5ZNJETJoB3x2UoyDbUMKo51LWxThPIV6w64wl2BV6dJF1shZihlNMZoHPv3XBAG3NmGt3Ep0e/KGUtWRoTh/i59Yi3DwIZMMkfoAl+F/DoUuzhTiXsCC+jciAhKcz8XSGT8lc+qaWAtYltiJQRSlzBTlXGk9Fq9hkMq6zOAFLNAFFkDMkmj2cpWBuJ9uo3L3JsW0iuHwoUJ/kE9keBB3fqwzCJYghtBocQG1Fd3kQOERG3QTZtmuwh6Nplr8AW4BFK+gGNuQZdzRq7Xh0U+QqADoYz+TULMBlIvtYuBpdNEUR3N0qvQXzWilJeQpgwkkTjO5m4IMMIHMfW5bgs1BTQVFTv47EkuhxCjNufBqIm4SM7Dn0E43tCCAFOi/w5fsavxAA+004Hfu9AqcfyduJExiNtj8ufAb+IPh50AAOHThf0kdidGadcLVyYKcK10cSSld3hA5MCMOyaZ4KVAYcFf7sAWgArcfekWsyXUGm5u/e4fTnUxDufzQNcr06i1szywmI5d3TgMp7P76qnUL1Qu8Ht22oDzx9f3p5c/77uNngtLykDXGIwzhPRQ8gM4bgp9JBWmyi6C/wCa2nTQrMhvvheNbdFtwhOoV9F923rFvdx2P/qECfZNoUecHPtg8twRRdcjhB9tj6AuQRFP5EcOKjytlnqh0dkKADagwlS6XzwU6ioXKcgoH3+x9N1SVa6TMSZD7fgJVBw4w7QaM2fx0094AA+P5OIGAL1cIoiAcHgVgFaUgHqqDgie/tZhcUrhI0l0Ay/qFJH9g1v9i+EakF3sDDA3DClUbBDtkSTzR1yMHGw8fbZAXnK5pP6ZnQo/PoAXBGcb9YpSGoFDD7fh5f3Zx/PP8P8LBd6ACakuuA6Bkc9DyiYW63tgu24TzYTF6kIkZJ1WEH7DiUQEHHdfyDvaMKGEK7ZjSBn+DTYMiXw0FQz2y7Xq9eENGJ2w3afa+M7QjO9dx31WyLmKHIxA5Rsn2INnD2WrokLS3gdV34uj78BzqxVi9WepBs94VON9QlLbesM+UfJ9YkJJsHGA2oyA2N5kscrebTbQJCvQlGR+1osiX4QsPkLmrhQQUgLCetMwPGI97thJNeYLO+a9fjLVbYCnOnNNKhkf8OJ31ShAa2YQ7IhpP9oJZa3CAgOpNRsIuyM19ZCiOYOUEe50jgV47FCERtCgrKn3GHsdbpb+wg7ex1upKQ5uBRfkLXE19M3iEbDAH0A24Pu43aHbiLGMFTdhuO6QU7mCRX8GFgY8dehPkJoELCLcyP+4r0M0F3MR8Irtj3DTODY9ryukf60qiRBhXmsbB5hX3NgXBTGY6cMoN1VcRFB2EyLJM30YTbRAQOoBmZn56sN+mfSoRNgp0irGgrpp5tH7arEGPivttt7OL+vmdXjF+gow/4DQsIHd8gpTFg6XNXHg4CDLJHSfTkoz/DW4+cVZS0SCbaNlvuHZg8pxYIkLvhGkOhFHXJUDYW8ztLcD2VzIQZ2poYtg3XUkFkfIDQVVyyTLMZ6CzaN6Y61quuN9imo3S9ZtMd9ugbNNuIY+N//aZGTG/TNMtlgNTruioiSt9FCBS+f1MebfLSypTe4ZqIn3gt6MnK4UhWiifa9rCYXtgwbd/r9gZyHj3eWag5fZAwcXBYf/B99ESwlhkx1xIewuAxMdGMU4SjWctMN892rMg7Gu2aoTeqXU73oFeZ2jfMtbqZugdHox3U8Y7q4BWWWz0JygCLFfWUHAhZawYPGQUuF/EGv2I2DIwaLm1tzYmuSqsIDbNMnGev0+q9c1A1dgiSbe/tSWA8+iOfgEX5rizrwSvAECo5Txb1wpM53854Gugpzpdon8HBifNyHohn7bg3ylNIzczKwofIQsGoybIt5v9I2PTesQJR1eFuk2aZz1fVpvW3tcBuIXsooDnh4IJi+CICwStG83Eyll4MlUJbjW53BwMFgR/wSRj4LcLFQZAObpHd1vYyDPZaXqfreW1YMPwyBBqHYYoxVIa1QmxhSqySvCqJOIQihc+P1vt4sYg2mDGT6cYsvktCSrFSIAFc5qHlWbP0iVl9njpk8PsOE4N8UzDw6n+luPuQ+JMnEYaC+XkyofjVN571aRRnd44T4J1uRbwKQStHnwwOzZ0XcvQ1GV/El8MERLCfpRtobvG+FHbjX9HeYRvY9A0s5WazjYqR2SqFw9Gi7AdPc8iUx4FIfPCEB0908GyJmd3gCRCmZzgK75aSteCA4QJRZy40hfHfbTgHxHO0633fE+Hc/ve79kTXgRGVR/J5+0MxJZ4f/pV5Pbva8Uh2HCCbAFeGGB8kg1e6phRWynJwgG63L2Q6Pa/DJOP5VeqhCYYvUXy3xPzsJt0mc2eT3pJj8phueCTmNspymU1+WoKrIuZEuPMYd0Yl0QtZMHuZrSJ/wnd34gas8s2raetpTwPDRvqTrU3finZ/MuSykROETBtiTBvcLc6tk3XgEy6TP/+Nj+m7HSgJvWYUx+SdqxI491WUV9+IHNzdjxdnv47fN8m8Ral87HuHaH23qj4Z9RGmu3CpbNCseN4Rv9Cwl6BRCBf0KGpEiwhi/BMJg2lyYQQpK3nSpNaS/JkyEH3TaoDvNfuLk2gNtjqcy0nrmb3wOMBLYd56Lm96NptiFU2nUXaZvARK+GFGBjLQpuyiDGlpEfv1sCVC9qUwv80wPs8wPC9OvBCx/mRdJC85VMQNOx7zsCEOPB6gqcnDjTVzlmE/LSPwlzLYfDFoMjwI7NFDtEoTo/lg6O1rkX8Nie8y5A6e00PaktsKV7Xiz+4GJJj3Pk7mTbUToH7HsEWsJubAhdWx3+dEw95Ehv/RqoczmZUl6qqnImHD03RavCyLTJ9I6JdJUAmYzITBhVJgSjYC4wYDW4Wg5JVzWz3pJSNmQoYWesWJrRxkcjU5PGlwQTOaW8GuAA0Htzt0ACo+91ukNKgn2CsHA2ndaBE13cTRmoOJZm0dgdizlYtbqkgQ8QZ7VxAAdCyFcRMx+YzpviA57MleT9Ai56QQKJMB5eBKKlDnotDI510njhdgPhCx9bi75cPEe7JbHRVnpXQwjUI7e39Q6U6hVRwg8+eqszeog408uk8sOisKBdSYQe0YkpYqj3bi8v4dH4mgFgK7WBn7o3X6mMZz689UJHlAGa82UTh/4QZ3pJVoUTkW2BAYkldlUrQx1p/bLAeFG2F5WbcaMEJk9nywXt3+oK1kuMK3bkXGRgt+MvbLDqwTy3Pdnju0BHW6R736uTMZNuC2HJvZdl0yj/etPxcz8nh4D1smi0nOCxE2a5RDV8mrYSsQv+KsVgQGX2ehjYC1MQRjkES4EjI8VFXUyPpTClSj0AxfGO2UjMGjcuPiE/vFq/ivCGN38I0ig8zxbB7fEYD52dRjioA4dnDKMcWGHtLaEVLUAlAp2Gpgak9C4ZuWiUbxAZBn4bEPHx0uu4lwTkXM8a4dT+oCAdxcgxEGQorAPG2ywnhv2xHNhkjTiezK5Co4i9OQnFXuSXAbiGcg32QJCXunZPyUDSTMhflallJE/RDJctqEt6LZlGYxIeZPFnx6mrEDP5ZAQewSJ49gRtWm8HlCdSYAYG6Pk0hBtUdotdRXCYwwlQd6cAQnJdo8oh8FfMYP3EOcZWjSfF0PQ93U5ZSTClCeR8NqlRaGbgDItpIVqzI7aYZqEzVIqdahlBCkmgYG3U1zvliuFqxNHrG0K5vEBd+BdwsWLE6DZQ5SqhVBV1VNKAoAuS3BanWhVoJHxVqFoEJyo+GYZiXTMc30XhGK5Y3fyrfrFS3B5oml8XuiGeF64lJOkzwEjx8phbrEDtQIOHO82IM/4Nhj+77RzhdH7TWIiE+aT27JyeFAQwaGGUn3I6OSV9T5UnQhpPIf5AdZ8gvbBGo+CxeRI8tGtquoW7L5fAw+yXwm4l1kM91hDaJ4znh2FstZXWf++rIWRcp+f+DqKyu5ymIMdkD+xrinpGIoOUYvYSmZguJEAuv5k+Z7kJFNEo81O80L3bDuqZ5dqmZ2eer6ahcN2Xo1yZ3eTPksjA6AOPb6+QbbJhR2uJDWUlwEdTMLuPWTrrXwRMYMx0cdGXCwlEDqhnP0uEZ1fgt3vIC+P/12/vH99OwCqEyKBWwxHncSDgF/fnl6ffPb1VhshKyE4F0mzcuPp2fjJgsDTd9yd8X3W83dsCVUWwJ6f/5LM9jFC5hS1rnA99clBoC13oJRd29oSgWgSlSU4CrToolIikogRbGDxpwwh6+EpS4ZTe4UG00cKk+ZN6hOX2GicAcTVXiHH3Jgc8WmJ66Bzau89FY+Cn2BC9h9mk1hMhdiMQkDx/dGxrm9PD/79bdLYArmBaQqu/xGiCR0Hmb36EaOwscwXoW3q6isZnlkxB5tkwUoV3dEll4CrZGe/BdHj1vab44hqVqe8iHEzToG0UyJU4mZ7nFULG9aiDK814yOw2dQbLOAee5AiWZRErnfYzgxkleCB8vMKaX365z6t9YcVM8gjxUMq5jSecNrEgKlOljfzcS/xVoeLaMV5eX17HklL1/rNKt5luEGC858BHXC1U+rJfPT9Iue+D2Vh+8d2twAQOT1noDmid/qSRdXxEOEyX3sia3S0ve2WVIlUOGzatl/WOATMC+Y17IYzeJO0okOy+ShOXihqziJprdpss38lue5nd6gvSN6ZuuqviacUN3jD6dXv1PRLvOwlB5dzJ7LkFRt5RjsHdodE4/XOKJCSb3OQSp9nGAnx+27r8En4VWlJBEbWQ1kTDTbYi4cGEn04pQ22AHNDvKTUAXXe12vsUfPLm3UE8Xq/cFAw3EXLjzT4Ll14uXEB2XAgbX9/nd278vpDfpljPcXVCsuNkRJtIhzf5cTrFNV9D0hc2/A6ry/dtcb2OoY+L1BqSqqEMAdEPdVZBUsQPidS7wGNGBi5r2BzYxCtULjvjBRk1U4JUVBd6l665nlZk8Yar+xaBl3W9VhvV5btPYp8l1rpIGnWeJLpFfvXVU5kO7aRSyQyQHr8SwX1zv7Qkccuuo61C4MZuGmHofqTOBp4Y6AMirzj6kuSpVN1dVURFXP7fTdfyqq/o64KmsOKbrq6uUwf7VToO1YuooObaZKLTeLo9A7qiPrxceP47ObqX6C2KGrT0SR9Sls1SbG+IDmOcvsufCGxE0D9LhtbujUDtKD0jUjv2/B6oaqMmMNi7XWniA2PjFWY5qg2nKkAhDRg9Jjk6kSskQHjOA7BnxGhqUsKRB2pvzp+MnI6C3jg7UupDRFJRZJrUGqYtiamDvRt6K04tJe1C7b7FO39j7TpnP06QQBdO6SVNDbkBT6sLdRQkesnhx/b9dNEb1WzFRjHJc2460MwOVX8/PF1BCM/3S//gHCOoC/v3UkGhD9qgbkiX3cAx7uQxS0y4to89J1TB1jOuE4pFq4fM8A73QtTQaQyaBbZpFtak2axa4tbpYjwDJW5B7uKjTdccx3Jt3oiJkzaBR52zSlrXzbXLUnVD58FYwMxqnApJKc5YgKD+1dcRcZiI13QEm182o/Pd6HpgkoKCtd0O34Nb1QAS/HW/kyxAKSDcav9Sv45YmW6Sb+K01ADb9zcI6Kuu3tU3tt3kmu6cQXYHYvn6cKOTdhbZYc2u4e9qqwZ8sUNLMvMlCO7MycmDn3tahQjYuMKGEsmSCcYPOQmJ83NHYMGZohnymbsiRmyb2PnUax7yTx6B7+34/oVMBzn3h/cl8ENXZHmQolWQk31XKBKUlHIsqRFzcKqIWmmLjBv/x10HjlSGPKk+t/4lgVzuVyokaRSzFWE0I1lVxNtNicQgdRXXE51FqgYJu1AxoUuvip5ZcmIe42thZRWdE9EKknnvGZppt5tKnmnpjo/bYc1N/MPo1ezzzxPGGYLf0iKj7ieGJQi8IdPJ5VJIwoOePLFI2ekglN2WyuSwYygV9AXtJmDmuKL8i3XZcyXLilMiiGOOFtVPws30XFgD2Zmp5xQ0b5YMaNmNeKXvRLLrJ0k6ThlJfiuBUpxS+HIQKdvopdT6k2Q3bGiEwR0+HD+qrKMvO/itvf3oF8UcYQL+4bpQv42g55CR1fx8Gv8+MXWa+Ag+UrAfoDQ98PPVd5ZDC6dNmgnPcy30YhkmD89RnV7Jeuy6TPV2xvYu5Wo1SbITDipmTCREEeMpga4hj0tysg9FW+CQ7ujK1nsj6kK/CSomi9oje4QL8Myym2+XqbyzoKPCn09hgylkZIOyyZ1BUiZb7wzkrvoGveGSlKnsTbO4zbbspHO+DFJDDTyYFIT6kA1zExCqbAQWnBOl0j3mQeJ36KpQ04aV6PP8KuwgGDI4zQwcJu80v7RWK9e6QX5MobqCRT9Pu/P1o/o6Km66pIDqwhT3D14Woo3rEkboT+Xry6aB7l0SxHwy56wbf30H1cDqNbTvVPMKHzx/Ty6uL9b2c3hZXt9YKgAw8/nGMoIGjvdybGmvkw7k0jd2K9shewUrMoT+5XHvCsDo6ogXpNhWryIOzLwaJZIHgQyMuNWYyBDcre+0dvLwD23l4A3NO26rg3NKfktUNSR6K8KXXAI2K04Pt8Gnr6QLuRSruEKsK4oTxaxLf+xGOeWWc71Qoz9+0hdJJMCF+xHKhDnz29LN8cxg+ugZ0jaCLKIexyVBNLp3yESxhWkpI4mh8IuhFL9hmcgWMc1jl0S8ZX3fHhLMePjkOVWiOaSt5o5E4fCgbfUAXGJnFDpZBIhadxXIzXxSbuUvHEqRuJ10EExQi3I9fe22vJuKi8gNYvhf4TkXrQCAPqaVhede0pTBQZknbNRKLQH5nsNUYeYRfaNnzjDm0SjZFHoae9hoeeHPPSh96xz0ul+0o4HXua0JRgO1J6lpbY2LXDuNaPp5/fF7ssQelHesfbK4pEXprVFcnI0zcJ1Ysg6LV29DaMYltr6wqqr7/4u+++eIuZM+T41L7iQmEk7lgIxPSk6/Owelfj6/1wsvumOHgu6lr0PQLf3fWbPXkOzDsc2srxhpC6chxiQmR0u32ZysApHSLvkPXeOdTVKZd5yI05ERQIi0C0rwMqeEybzA06YJO9ymi7mE3qHcwtS5YzII8UQnIjEqq+/2qy29syx6mZR2kZzmrrLUXbMgMrHDdx8oaEE1rkiVHW2vHKN2wF8/BuAlOdhe4fhzypdf+Ib73BqtGyLORyLnHWKrGu5qsIQtgWkIQqTeYGJQn4VvnHNfzMEHsa1IaRU9JS+v/fh7UoxKffu5Jl382MDRWykqu0W0PeUREO7PBXQqnfekpNz4TId1HURbWOOexXz0eiFXI4OyFVtndgKDp9mVVt9x3FZsaRjbt6fOAEXCfpzdM7KLUXZFTe2mPe8xaXln3j1RraO3D83RcAuZqgCzTfuchjlIC+XumqL45ce15u6ssQhxswbiHKBm8YMC6V/TcGMuzG/wFrIrK40lUAAA=='''
pathlib.Path('main.py').write_bytes(gzip.decompress(base64.b64decode(blob)))
compile(pathlib.Path('main.py').read_text(), 'main.py', 'exec')
print('main.py written and compiled')


## Head-to-head: v18 vs exact repository v17
Runs both seats over 12 seeds. This is live reaction testing, not replay lookup.


In [ ]:
import json, urllib.request, pathlib
from kaggle_environments import make
u='https://raw.githubusercontent.com/prince22466/Co_AMAP/main/Co_Kaggle/g5/submission_nb/kaggriculture-sub_v17.ipynb'
nb17=json.loads(urllib.request.urlopen(u).read())
src=next(''.join(c.get('source',[])).split('\n',1)[1] for c in nb17['cells'] if ''.join(c.get('source',[])).startswith('%%writefile main.py'))
pathlib.Path('v17.py').write_text(src)
W=L=T=0; margins=[]
for seed in range(20260910,20260922):
  for pair,me in [(['main.py','v17.py'],0),(['v17.py','main.py'],1)]:
    e=make('kaggriculture',configuration={'episodeSteps':720,'seed':seed},debug=False); e.run(pair)
    r=[s.reward for s in e.steps[-1]]; m=r[me]-r[1-me]; margins.append(m); W+=m>0; L+=m<0; T+=m==0
print({'wins':W,'losses':L,'ties':T,'mean_margin':sum(margins)/len(margins),'min_margin':min(margins)})


## v16/v17 loss-history inventory
Run from a repo checkout to ensure both historical loss folders are present before replay regression. The submission agent never reads them.


In [ ]:
from pathlib import Path
for d in ['Co_Kaggle/g5/game_history/v16','Co_Kaggle/g5/game_history/v17','../game_history/v16','../game_history/v17']:
  p=Path(d)
  if p.exists(): print(d, len(list(p.glob('*.json'))), [x.name for x in sorted(p.glob('*.json'))])


## Standard validation and submission archive


In [ ]:
from kaggle_environments import make
for i,opp in enumerate(['main.py','starter','random']):
  e=make('kaggriculture',configuration={'episodeSteps':720,'seed':20260940+i},debug=True); e.run(['main.py',opp]); f=e.steps[-1]; print(opp,[s.status for s in f],[s.reward for s in f])
import tarfile
with tarfile.open('submission.tar.gz','w:gz') as z: z.add('main.py',arcname='main.py',recursive=False)
print('submission.tar.gz created')
